# ARCADE Stenosis Detection v4 — Root-Cause Fix After the v2 Run

## What actually happened in v2 (read this before running anything)

v2's real Colab run came back with: **Precision 0.274, Recall 0.365, F1 = 0.313, mAP@0.5 = 0.244**.
That is far below both the literature (StenUNet F1 ≈ 0.535) and a plain community YOLOv8 baseline on this same
dataset (mAP@0.5 ≈ 0.727). The gap is too large to be "just a hard problem" — something in v2's pipeline made
things actively worse, and it needed to be found before doing anything else (v3, which added SAHI tiling and a
flip-TTA ensemble on top of the same pipeline, is **shelved** until this is fixed and validated).

**Root cause (most likely): the 3-channel preprocessing broke transfer learning.**

v2 stacked three *different* filtered views into one image — channel 0 = denoised grayscale, channel 1 = CLAHE,
channel 2 = a Sato vesselness map — as the model's R/G/B input. Look at the "Ground Truth" / "Prediction" images
from your actual run: they're a flat green/teal composite, nothing like a photograph. That matters because
`yolov8s.pt`'s pretrained weights were trained on real photographs (COCO), and the backbone's early convolution
filters are tuned to that kind of input. Feeding it three decorrelated filter outputs stacked as color channels is
badly out-of-distribution for those pretrained filters — on a dataset this small (~1000 training images), the
model leans heavily on that pretrained initialization, so breaking it plausibly explains a 3x performance gap
against a plain-image community baseline. Aggressive non-local-means denoising and the vesselness filter on top of
that may also have suppressed some of the faint stenosis signal itself, compounding the problem.

**Fix applied in this notebook**: preprocessing is now CLAHE-only, applied to grayscale and replicated to 3 identical
channels. This is the standard, well-established way to feed medical grayscale images to an ImageNet/COCO-pretrained
CNN — it boosts local contrast (still useful for faint lesions) while keeping the image visually a normal-looking
photo, so the pretrained backbone's filters still apply. `imgsz=960` is kept — that part of v2 was justified by the
lesion-size measurement (Section 4 below), independent of the preprocessing bug, and isn't in question here.

**Scope is deliberately narrower than v3.** Per your supervisor's Mid-Defense document, Objective 1 is: a working
detection/localization model on ARCADE, evaluated with Dice/IoU/F1/Sensitivity/PPV, benchmarked against U-Net,
Attention U-Net, and a detection baseline. This notebook gets the YOLOv8 detection baseline itself right and
honest — trained twice (raw images vs. CLAHE-enhanced) in one run, so the preprocessing question is answered with
real numbers on your own data, not assumed. It does **not** add SAHI/TTA/ensembling back in — that speculative
complexity is exactly what compounded the problem last time, and shouldn't be layered on until this baseline is
confirmed solid. The U-Net and Attention U-Net baselines your proposal also commits to are a separate, sizeable
notebook of their own — flagged at the end, not attempted here alongside an unvalidated detector.

**What to do with this notebook**: run it once, end to end. It trains Approach RAW and Approach CLAHE back to back
and prints both test-set results next to each other. Compare both against the 0.313 F1 / 0.244 mAP@0.5 you already
have from v2, and against the 0.727 mAP@0.5 community number. If CLAHE beats RAW and both clearly beat v2's numbers,
that confirms the diagnosis and you have a validated baseline to build on. If not, say so and paste the results back
before we go any further — do not let me guess again without checking against your actual run.


## 1. Install & Import Dependencies

In [ ]:
# ============================================================
# 1a. Installs
# ============================================================
!pip install -q ultralytics pycocotools scikit-image kagglehub
print("Installs complete.")


In [ ]:
# ============================================================
# 1b. Imports
# ============================================================
import os
import json
import shutil
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm

import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from ultralytics import YOLO, settings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

settings.update({"raytune": False})   # avoids a spurious Ultralytics/ray version mismatch crash on Colab


## 2. Get the ARCADE Dataset onto the Colab Runtime

In [ ]:
# ============================================================
# 2a. Download dataset via kagglehub
# ============================================================
import kagglehub

download_path = Path(kagglehub.dataset_download("nirmalgaud/arcade-dataset"))
print("Downloaded to:", download_path)

candidates = list(download_path.rglob("train.json"))
if not candidates:
    raise FileNotFoundError(
        f"Could not find train.json anywhere under {download_path}. "
        "Browse the folder manually with `!find {download_path}` and set DATASET_ROOT by hand."
    )

DATASET_ROOT = candidates[0].parent.parent.parent
print("Resolved DATASET_ROOT:", DATASET_ROOT)

for split in ["train", "val", "test"]:
    ann = DATASET_ROOT / split / "annotations" / f"{split}.json"
    img_dir = DATASET_ROOT / split / "images"
    assert ann.exists(), f"Missing {ann}"
    assert img_dir.exists(), f"Missing {img_dir}"
print("Dataset structure verified for train/val/test.")


In [ ]:
# ============================================================
# 2b. Mount Google Drive so trained weights / results survive the session
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/arcade_stenosis_runs")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Drive output dir:", DRIVE_OUTPUT_DIR)


## 3. Configuration

In [ ]:
# ============================================================
# 3. Configuration
# ============================================================
WORK_ROOT = Path("/content/stenosis_yolo_v4")
PREPROC_ROOT = WORK_ROOT / "preprocessed_images"
YOLO_ROOT = WORK_ROOT / "yolo_dataset"

SPLITS = ["train", "val", "test"]
STENOSIS_CATEGORY_ID = 26
CLASS_NAMES = ["stenosis"]

# Both are trained automatically in this notebook -- no manual re-run needed.
RUN_VARIANTS = ["raw", "clahe"]

IMG_SIZE = 960          # unchanged from v2 -- justified by lesion-size measurement in Section 4, not in question here
BATCH_SIZE = 16
EPOCHS = 150
PATIENCE = 40            # raised from v2's 25: the v2 training curve (recall/mAP) was still slowly climbing when
                          # early stopping fired -- giving it more room removes that as a confound in this comparison
MODEL_WEIGHTS = "yolov8s.pt"

if torch.cuda.is_available():
    DEVICE = 0
else:
    DEVICE = "cpu"
    print("WARNING: no GPU detected -- this notebook trains two models; on CPU this will be very slow. "
          "Go to Runtime > Change runtime type > T4 GPU.")

for p in [WORK_ROOT, PREPROC_ROOT, YOLO_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("Work root:", WORK_ROOT, "| Device:", DEVICE, "| Variants to train:", RUN_VARIANTS)


## 4. Explore the Dataset & Quantify Lesion Size

Unchanged from v2 -- this measurement (not the preprocessing) is what justifies `IMG_SIZE=960`, so it's kept as-is
and re-verified here.


In [ ]:
# ============================================================
# 4. Load COCO annotations, quantify lesion size relative to image
# ============================================================
def load_coco(split):
    ann_path = DATASET_ROOT / split / "annotations" / f"{split}.json"
    with open(ann_path, "r") as f:
        return json.load(f)

coco_data = {split: load_coco(split) for split in SPLITS}

summary_rows = []
for split, coco in coco_data.items():
    n_images = len(coco["images"])
    n_anns = len(coco["annotations"])
    n_imgs_with_ann = len(set(a["image_id"] for a in coco["annotations"]))
    summary_rows.append({"split": split, "num_images": n_images, "num_annotations": n_anns,
                          "num_images_with_annotations": n_imgs_with_ann,
                          "pct_images_with_stenosis": round(100 * n_imgs_with_ann / n_images, 1)})
print(pd.DataFrame(summary_rows).to_string(index=False))

size_rows = []
for split, coco in coco_data.items():
    img_wh = {im["id"]: (im["width"], im["height"]) for im in coco["images"]}
    for ann in coco["annotations"]:
        if ann["category_id"] != STENOSIS_CATEGORY_ID:
            continue
        img_w, img_h = img_wh[ann["image_id"]]
        _, _, w, h = ann["bbox"]
        size_rows.append({"split": split, "box_area_pct_of_image": 100 * (w * h) / (img_w * img_h)})

size_df = pd.DataFrame(size_rows)
median_pct = size_df["box_area_pct_of_image"].median()
print(f"\nMedian stenosis box covers only {median_pct:.2f}% of the image area -- basis for IMG_SIZE=960.")


## 5. Preprocessing (Fixed): CLAHE Only, Kept Photographic

`build_raw_image` returns the original image unchanged. `build_clahe_image` applies CLAHE to the grayscale version
and replicates it to 3 identical channels -- contrast-enhanced, but still a normal-looking grayscale photo, so
`yolov8s.pt`'s pretrained filters still apply. No denoising, no vesselness channel, no channel stacking of different
filters -- those are exactly what v2's evidence points to as harmful here.


In [ ]:
# ============================================================
# 5a. Preprocessing functions
# ============================================================
def to_grayscale(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img

def build_raw_image(img_bgr):
    return img_bgr

def build_clahe_image(img_bgr, clip_limit=2.5, tile_grid_size=(8, 8)):
    gray = to_grayscale(img_bgr)
    gray = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    enhanced = clahe.apply(gray)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)   # replicate to 3 identical channels -- still photographic

PREPROCESS_FN = {"raw": build_raw_image, "clahe": build_clahe_image}

def imread_any(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    return img


In [ ]:
# ============================================================
# 5b. Visual sanity check -- both variants should look like normal X-ray photos,
#     unlike v2's green/teal composite.
# ============================================================
sample_img_dir = DATASET_ROOT / "train" / "images"
sample_files = sorted(os.listdir(sample_img_dir))[:3]

fig, axes = plt.subplots(len(sample_files), 3, figsize=(12, 4 * len(sample_files)))
for i, fname in enumerate(sample_files):
    img = imread_any(sample_img_dir / fname)
    raw_img = build_raw_image(img)
    clahe_img = build_clahe_image(img)
    for ax, im, title in zip(axes[i], [img, raw_img, clahe_img], ["Original", "RAW variant", "CLAHE variant"]):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
plt.show()
print("Both variants should look like ordinary grayscale angiograms -- no green/teal color cast.")


In [ ]:
# ============================================================
# 5c. Apply both variants to every split
# ============================================================
def preprocess_split(split, variant):
    src_dir = DATASET_ROOT / split / "images"
    dst_dir = PREPROC_ROOT / variant / split
    dst_dir.mkdir(parents=True, exist_ok=True)
    fn = PREPROCESS_FN[variant]
    for fname in tqdm(sorted(os.listdir(src_dir)), desc=f"Preprocessing {split} ({variant})"):
        img = imread_any(src_dir / fname)
        cv2.imwrite(str(dst_dir / fname), fn(img))

for variant in RUN_VARIANTS:
    for split in SPLITS:
        preprocess_split(split, variant)

print("Preprocessing complete for variants:", RUN_VARIANTS)


## 6. COCO → YOLO Conversion

In [ ]:
# ============================================================
# 6. COCO bbox -> YOLO bbox conversion
# ============================================================
def coco_bbox_to_yolo(bbox, img_w, img_h):
    x_min, y_min, w, h = bbox
    x_center, y_center = x_min + w / 2.0, y_min + h / 2.0
    return (x_center / img_w, y_center / img_h, w / img_w, h / img_h)

def convert_split_to_yolo_labels(coco, label_out_dir):
    label_out_dir.mkdir(parents=True, exist_ok=True)
    images_by_id = {im["id"]: im for im in coco["images"]}
    anns_by_image = defaultdict(list)
    for ann in coco["annotations"]:
        if ann["category_id"] == STENOSIS_CATEGORY_ID:
            anns_by_image[ann["image_id"]].append(ann)

    n_boxes, n_empty = 0, 0
    for img_id, img_info in images_by_id.items():
        stem = Path(img_info["file_name"]).stem
        img_w, img_h = img_info["width"], img_info["height"]
        lines = []
        for ann in anns_by_image.get(img_id, []):
            xc, yc, w, h = coco_bbox_to_yolo(ann["bbox"], img_w, img_h)
            lines.append(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
        with open(label_out_dir / f"{stem}.txt", "w") as f:
            f.write("\n".join(lines))
        n_boxes += len(lines)
        n_empty += (len(lines) == 0)
    print(f"  {len(images_by_id)} images, {n_boxes} boxes, {n_empty} empty/background labels")


## 7. Assemble YOLO Datasets (one per variant) + Augmentation Config

In [ ]:
# ============================================================
# 7a. Assemble a YOLO dataset dir per variant, sharing one set of label files
#     (labels are identical across variants -- only the images differ)
# ============================================================
yaml_paths = {}

for variant in RUN_VARIANTS:
    root = YOLO_ROOT / variant
    for split in SPLITS:
        img_dst = root / "images" / split
        lbl_dst = root / "labels" / split
        img_dst.mkdir(parents=True, exist_ok=True)

        src_dir = PREPROC_ROOT / variant / split
        for fname in os.listdir(src_dir):
            shutil.copy(src_dir / fname, img_dst / fname)

        print(f"[{variant}/{split}]", end=" ")
        convert_split_to_yolo_labels(coco_data[split], lbl_dst)

    yaml_path = WORK_ROOT / f"data_{variant}.yaml"
    with open(yaml_path, "w") as f:
        f.write(f"path: {root}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: stenosis\n")
    yaml_paths[variant] = yaml_path

print("\ndata.yaml files:", {k: str(v) for k, v in yaml_paths.items()})


In [ ]:
# ============================================================
# 7b. Augmentation
#     hsv_s / hsv_v restored to mild, non-zero values (unlike v2): both variants here are
#     photographic 3-channel images (grayscale-replicated or original), so brightness/contrast
#     jitter is physically meaningful again, not noise on a synthetic filter stack.
#     Geometric augmentation rationale is unchanged from v2: small jitter for generalization,
#     no vertical flip / perspective since those are anatomically implausible for this view.
# ============================================================
augmentation_params = dict(
    hsv_h=0.0, hsv_s=0.1, hsv_v=0.2,
    degrees=10.0, translate=0.10, scale=0.30, shear=2.0, perspective=0.0,
    flipud=0.0, fliplr=0.5,
    mosaic=0.8, mixup=0.1, close_mosaic=10,
)
for k, v in augmentation_params.items():
    print(f"{k:12s}: {v}")


## 8. Train Both Variants

Trains RAW then CLAHE, back to back, in the same run -- no manual re-configuration needed. Same architecture,
hyperparameters, seed and augmentation for both, so any difference in the final numbers is attributable to the
preprocessing variant alone.


In [ ]:
# ============================================================
# 8. Train loop over both variants
# ============================================================
run_dirs, best_weights = {}, {}

for variant in RUN_VARIANTS:
    print(f"\n{'='*60}\nTraining variant: {variant}\n{'='*60}")
    model = YOLO(MODEL_WEIGHTS)
    train_results = model.train(
        data=str(yaml_paths[variant]),
        epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE, device=DEVICE,
        project=str(WORK_ROOT / "runs"), name=f"stenosis_{variant}",
        patience=PATIENCE, optimizer="auto", lr0=0.01, cos_lr=True, seed=SEED,
        pretrained=True, verbose=True, plots=True, **augmentation_params,
    )
    run_dirs[variant] = Path(train_results.save_dir)
    best_weights[variant] = run_dirs[variant] / "weights" / "best.pt"
    print(f"[{variant}] run saved at:", run_dirs[variant])


## 9. Training Curves, Both Variants Side by Side

In [ ]:
# ============================================================
# 9. Training curves
# ============================================================
fig, axes = plt.subplots(1, len(RUN_VARIANTS), figsize=(8 * len(RUN_VARIANTS), 6))
if len(RUN_VARIANTS) == 1:
    axes = [axes]
for ax, variant in zip(axes, RUN_VARIANTS):
    img_path = run_dirs[variant] / "results.png"
    if img_path.exists():
        ax.imshow(plt.imread(img_path))
    ax.set_title(f"Variant: {variant}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 10. Evaluate Both Variants on the Held-out Test Set

Note for the thesis writeup: **Recall = Sensitivity** and **Precision = PPV** — same numbers, medical-literature names.


In [ ]:
# ============================================================
# 10a. Evaluate each variant's best.pt on its own test split
# ============================================================
test_metrics = {}
manual_metrics = {}

def iou_xyxy(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b
    ix1, iy1 = max(xa1, xb1), max(ya1, yb1)
    ix2, iy2 = min(xa2, xb2), min(ya2, yb2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    area_a, area_b = (xa2 - xa1) * (ya2 - ya1), (xb2 - xb1) * (yb2 - yb1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def yolo_line_to_xyxy(line, img_w, img_h):
    cls, xc, yc, bw, bh = map(float, line.split())
    return [(xc - bw / 2) * img_w, (yc - bh / 2) * img_h, (xc + bw / 2) * img_w, (yc + bh / 2) * img_h]

def manual_iou_eval(model, img_dir, lbl_dir, conf_thres=0.25, iou_thres=0.5):
    TP, FP, FN = 0, 0, 0
    for fname in tqdm(sorted(os.listdir(img_dir)), desc="manual IoU eval"):
        stem = Path(fname).stem
        lbl_path = lbl_dir / f"{stem}.txt"
        img = cv2.imread(str(img_dir / fname))
        h, w = img.shape[:2]
        gt_boxes = []
        if lbl_path.exists() and os.path.getsize(lbl_path) > 0:
            with open(lbl_path) as f:
                gt_boxes = [yolo_line_to_xyxy(l.strip(), w, h) for l in f if l.strip()]
        result = model.predict(source=str(img_dir / fname), imgsz=IMG_SIZE, conf=conf_thres, verbose=False)[0]
        pred_boxes = [box.xyxy[0].cpu().numpy().tolist() for box in result.boxes]
        matched = set()
        for pb in pred_boxes:
            best_iou, best_idx = 0.0, -1
            for idx, gb in enumerate(gt_boxes):
                if idx in matched:
                    continue
                cur = iou_xyxy(pb, gb)
                if cur > best_iou:
                    best_iou, best_idx = cur, idx
            if best_iou >= iou_thres:
                TP += 1
                matched.add(best_idx)
            else:
                FP += 1
        FN += len(gt_boxes) - len(matched)
    precision = TP / (TP + FP + 1e-16)
    recall = TP / (TP + FN + 1e-16)
    f1 = 2 * precision * recall / (precision + recall + 1e-16)
    return {"TP": TP, "FP": FP, "FN": FN, "Precision": precision, "Recall": recall, "F1": f1}


for variant in RUN_VARIANTS:
    print(f"\n--- Evaluating variant: {variant} ---")
    best_model = YOLO(str(best_weights[variant]))
    tm = best_model.val(
        data=str(yaml_paths[variant]), split="test", imgsz=IMG_SIZE, device=DEVICE,
        project=str(WORK_ROOT / "runs"), name=f"stenosis_{variant}_test_eval", plots=True,
    )
    test_metrics[variant] = tm

    test_img_dir = YOLO_ROOT / variant / "images" / "test"
    test_lbl_dir = YOLO_ROOT / variant / "labels" / "test"
    manual_metrics[variant] = manual_iou_eval(best_model, test_img_dir, test_lbl_dir)


In [ ]:
# ============================================================
# 10b. Combined results table -- RAW vs CLAHE vs v2's actual numbers vs literature
# ============================================================
rows = []
for variant in RUN_VARIANTS:
    tm = test_metrics[variant]
    p, r = float(tm.box.mp), float(tm.box.mr)
    f1 = 2 * p * r / (p + r + 1e-16)
    rows.append({
        "Method": f"v4 -- {variant} (this notebook, Ultralytics val)",
        "Precision_PPV": round(p, 4), "Recall_Sensitivity": round(r, 4), "F1": round(f1, 4),
        "mAP@0.5": round(float(tm.box.map50), 4), "mAP@0.5:0.95": round(float(tm.box.map), 4),
    })
    mm = manual_metrics[variant]
    rows.append({
        "Method": f"v4 -- {variant} (this notebook, manual IoU-match)",
        "Precision_PPV": round(mm["Precision"], 4), "Recall_Sensitivity": round(mm["Recall"], 4),
        "F1": round(mm["F1"], 4), "mAP@0.5": "-", "mAP@0.5:0.95": "-",
    })

rows.append({"Method": "v2 (actual Colab run, enhanced/broken preprocessing)",
             "Precision_PPV": 0.274, "Recall_Sensitivity": 0.365, "F1": 0.313,
             "mAP@0.5": 0.244, "mAP@0.5:0.95": 0.077})
rows.append({"Method": "StenUNet (ARCADE challenge, 3rd place, own eval)",
             "Precision_PPV": "-", "Recall_Sensitivity": "-", "F1": 0.535, "mAP@0.5": "-", "mAP@0.5:0.95": "-"})
rows.append({"Method": "Community YOLOv8 benchmark (sota2.com, unspecified protocol)",
             "Precision_PPV": "-", "Recall_Sensitivity": "-", "F1": "-", "mAP@0.5": 0.727, "mAP@0.5:0.95": "-"})

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

comparison_path = WORK_ROOT / "v4_comparison_table.csv"
comparison_df.to_csv(comparison_path, index=False)
print("\nSaved to:", comparison_path)
print("\nIf CLAHE's numbers clearly beat both RAW and v2's 0.313 F1 / 0.244 mAP@0.5, that supports the root-cause "
      "diagnosis in Section 0. If CLAHE does NOT beat RAW, or both are still far below 0.727 mAP@0.5, something "
      "else is also wrong -- paste this table back before adding anything else to the pipeline.")


## 11. Visualize Predictions vs Ground Truth (CLAHE variant)

In [ ]:
# ============================================================
# 11. Sample visualizations for the CLAHE variant (change to "raw" to inspect that one instead)
# ============================================================
inspect_variant = "clahe" if "clahe" in RUN_VARIANTS else RUN_VARIANTS[0]
inspect_model = YOLO(str(best_weights[inspect_variant]))
test_img_dir = YOLO_ROOT / inspect_variant / "images" / "test"
test_lbl_dir = YOLO_ROOT / inspect_variant / "labels" / "test"

sample_files = random.sample(sorted(os.listdir(test_img_dir)), min(6, len(os.listdir(test_img_dir))))
fig, axes = plt.subplots(len(sample_files), 2, figsize=(10, 5 * len(sample_files)))
if len(sample_files) == 1:
    axes = axes[None, :]

for i, fname in enumerate(sample_files):
    stem = Path(fname).stem
    img = cv2.cvtColor(cv2.imread(str(test_img_dir / fname)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    axes[i, 0].imshow(img)
    lbl_path = test_lbl_dir / f"{stem}.txt"
    if lbl_path.exists() and os.path.getsize(lbl_path) > 0:
        with open(lbl_path) as f:
            for line in f:
                if not line.strip():
                    continue
                x1, y1, x2, y2 = yolo_line_to_xyxy(line.strip(), w, h)
                axes[i, 0].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                                         linewidth=2, edgecolor="red", facecolor="none"))
    axes[i, 0].set_title("Ground Truth"); axes[i, 0].axis("off")

    result = inspect_model.predict(source=str(test_img_dir / fname), imgsz=IMG_SIZE, conf=0.25, verbose=False)[0]
    axes[i, 1].imshow(img)
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0])
        axes[i, 1].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                                 linewidth=2, edgecolor="lime", facecolor="none"))
        axes[i, 1].text(x1, max(y1 - 5, 0), f"{conf:.2f}", color="lime", fontsize=9, weight="bold")
    axes[i, 1].set_title(f"Prediction ({inspect_variant})"); axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


## 12. Save Everything to Google Drive

In [ ]:
# ============================================================
# 12. Persist weights + comparison table
# ============================================================
for variant in RUN_VARIANTS:
    out_dir = DRIVE_OUTPUT_DIR / f"v4_{variant}"
    out_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(best_weights[variant], out_dir / "best.pt")
    results_png = run_dirs[variant] / "results.png"
    if results_png.exists():
        shutil.copy(results_png, out_dir / "results.png")

shutil.copy(comparison_path, DRIVE_OUTPUT_DIR / comparison_path.name)
print("Saved to:", DRIVE_OUTPUT_DIR)


## Notes & Next Steps

- **Run this, then report back the Section 10b table before anything else is changed.** That table is the evidence
  for whether the root-cause diagnosis in Section 0 was right. Don't let a claim about *why* something works
  substitute for checking that it actually did.
- **If numbers are still far below the 0.727 community mAP@0.5** even with CLAHE: the next things to check, in
  order of likely impact, are (1) whether that 0.727 number used a materially different train/val/test split or
  confidence threshold than the official ARCADE split used here -- re-read the sota2.com page's methodology notes
  if any are given; (2) `yolov8m.pt` instead of `yolov8s.pt` for more model capacity; (3) whether `PATIENCE=40` was
  actually enough, i.e. check the results.csv tail in Section 9 to see if recall/mAP were still climbing when
  training stopped.
- **U-Net and Attention U-Net baselines are still owed** per Objective 1 of the Mid-Defense proposal (the objectives
  document explicitly names them as comparison points, alongside Dice/IoU metrics this notebook doesn't compute).
  That is a separate, substantial notebook -- ask for it once this YOLO baseline is confirmed working, so it's built
  against a validated preprocessing choice rather than repeating today's mistake in a second pipeline.
- **SAHI tiling / flip-TTA (v3's ideas) are not deleted, just postponed.** Once this baseline is solid, revisiting
  small-object tiling is still a reasonable direction -- but only on top of a preprocessing choice that's been
  checked against real numbers first.
